In [1]:
# ============================================================
# RESIDE-6K: LOAD BEST MODEL + GENERATE RESTORED TEST IMAGES
# ============================================================

# -------------------- IMPORTS --------------------

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

from skimage.metrics import peak_signal_noise_ratio as sk_psnr
from skimage.metrics import structural_similarity as sk_ssim


# ============================================================
# 1. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)


# ============================================================
# 2. PATHS
# ============================================================

# RESIDE dataset
DATA_ROOT = "/kaggle/input/datasets/lakshmiksr123/reside/RESIDE-6K"

TEST_INPUT_DIR = os.path.join(
    DATA_ROOT,
    "test",
    "hazy"
)

TEST_TARGET_DIR = os.path.join(
    DATA_ROOT,
    "test",
    "GT"
)

# Your saved best model
BEST_MODEL_PATH = "/kaggle/input/datasets/lakshmiksr123/result6/best_model.pth"

# Kaggle working output directory
OUTPUT_DIR = "/kaggle/working/restored_results"


print("\nBest model:")
print(BEST_MODEL_PATH)

print("\nTest hazy folder:")
print(TEST_INPUT_DIR)

print("\nTest GT folder:")
print(TEST_TARGET_DIR)


# ============================================================
# 3. CHECK PATHS
# ============================================================

if not os.path.exists(BEST_MODEL_PATH):
    raise FileNotFoundError(
        "best_model.pth not found:\n" + BEST_MODEL_PATH
    )

if not os.path.exists(TEST_INPUT_DIR):
    raise FileNotFoundError(
        "Test hazy folder not found:\n" + TEST_INPUT_DIR
    )

if not os.path.exists(TEST_TARGET_DIR):
    raise FileNotFoundError(
        "Test GT folder not found:\n" + TEST_TARGET_DIR
    )

print("\nAll paths found successfully!")


# ============================================================
# 4. IMAGE LIST
# ============================================================

IMG_EXTS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp"
}


def list_images(path):

    return sorted([
        f
        for f in os.listdir(path)
        if os.path.splitext(f)[1].lower() in IMG_EXTS
    ])


# ============================================================
# 5. TEST DATASET
# ============================================================

class PairedTestDataset(Dataset):

    def __init__(
        self,
        input_dir,
        target_dir
    ):

        self.input_dir = input_dir
        self.target_dir = target_dir

        self.inputs = list_images(input_dir)
        self.targets = list_images(target_dir)

        self.to_tensor = transforms.ToTensor()

        print("\nInput images :", len(self.inputs))
        print("GT images    :", len(self.targets))

        if len(self.inputs) != len(self.targets):

            raise ValueError(
                "Input and GT image counts do not match."
            )


    def __len__(self):

        return len(self.inputs)


    def __getitem__(self, idx):

        input_name = self.inputs[idx]
        target_name = self.targets[idx]

        input_path = os.path.join(
            self.input_dir,
            input_name
        )

        target_path = os.path.join(
            self.target_dir,
            target_name
        )

        inp = Image.open(
            input_path
        ).convert("RGB")

        tgt = Image.open(
            target_path
        ).convert("RGB")

        inp = self.to_tensor(inp)
        tgt = self.to_tensor(tgt)

        return inp, tgt, input_name


# ============================================================
# 6. CREATE TEST LOADER
# ============================================================

test_dataset = PairedTestDataset(
    TEST_INPUT_DIR,
    TEST_TARGET_DIR
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0
)

print("\nTest dataset ready.")
print("Total test images:", len(test_dataset))


# ============================================================
# 7. DENSE BLOCK
# ============================================================

class DenseBlock(nn.Module):

    def __init__(
        self,
        in_ch,
        growth=16,
        layers=3
    ):

        super().__init__()

        self.layers = nn.ModuleList()

        ch = in_ch

        for i in range(layers):

            self.layers.append(
                nn.Sequential(
                    nn.BatchNorm2d(ch),
                    nn.ReLU(),
                    nn.Conv2d(
                        ch,
                        growth,
                        kernel_size=3,
                        stride=1,
                        padding=1
                    )
                )
            )

            ch += growth


    def forward(self, x):

        features = [x]

        for layer in self.layers:

            out = layer(
                torch.cat(features, dim=1)
            )

            features.append(out)

        return torch.cat(features, dim=1)


# ============================================================
# 8. CHANNEL ATTENTION
# ============================================================

class ChannelAttention(nn.Module):

    def __init__(self, ch):

        super().__init__()

        self.net = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),

            nn.Conv2d(
                ch,
                ch // 8,
                kernel_size=1
            ),

            nn.ReLU(),

            nn.Conv2d(
                ch // 8,
                ch,
                kernel_size=1
            ),

            nn.Sigmoid()
        )


    def forward(self, x):

        return x * self.net(x)


# ============================================================
# 9. U-NET DENSE ATTENTION
# ============================================================

class UNetDenseAttention(nn.Module):

    def __init__(self):

        super().__init__()

        base = 32
        g = 16

        self.enc1 = nn.Conv2d(
            3,
            base,
            3,
            1,
            1
        )

        self.enc2 = DenseBlock(
            base,
            g
        )

        self.enc3 = DenseBlock(
            base + 3 * g,
            g
        )

        self.enc4 = DenseBlock(
            base + 6 * g,
            g
        )

        self.pool = nn.MaxPool2d(2)

        self.att2 = ChannelAttention(
            base + 3 * g
        )

        self.att3 = ChannelAttention(
            base + 6 * g
        )

        self.att4 = ChannelAttention(
            base + 9 * g
        )

        self.bottle = DenseBlock(
            base + 9 * g,
            g
        )

        self.up4 = nn.Conv2d(
            base + 12 * g + base + 9 * g,
            128,
            3,
            1,
            1
        )

        self.up3 = nn.Conv2d(
            128 + base + 6 * g,
            64,
            3,
            1,
            1
        )

        self.up2 = nn.Conv2d(
            64 + base + 3 * g,
            32,
            3,
            1,
            1
        )

        self.final = nn.Conv2d(
            32 + base,
            3,
            1
        )


    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)

        e2 = self.att2(
            self.enc2(
                self.pool(e1)
            )
        )

        e3 = self.att3(
            self.enc3(
                self.pool(e2)
            )
        )

        e4 = self.att4(
            self.enc4(
                self.pool(e3)
            )
        )

        # Bottleneck
        b = self.bottle(
            self.pool(e4)
        )

        # Decoder
        d4 = F.interpolate(
            b,
            size=e4.shape[2:],
            mode="bilinear"
        )

        d4 = torch.cat(
            [d4, e4],
            dim=1
        )

        d4 = F.relu(
            self.up4(d4)
        )

        d3 = F.interpolate(
            d4,
            size=e3.shape[2:],
            mode="bilinear"
        )

        d3 = torch.cat(
            [d3, e3],
            dim=1
        )

        d3 = F.relu(
            self.up3(d3)
        )

        d2 = F.interpolate(
            d3,
            size=e2.shape[2:],
            mode="bilinear"
        )

        d2 = torch.cat(
            [d2, e2],
            dim=1
        )

        d2 = F.relu(
            self.up2(d2)
        )

        d1 = F.interpolate(
            d2,
            size=e1.shape[2:],
            mode="bilinear"
        )

        d1 = torch.cat(
            [d1, e1],
            dim=1
        )

        # Residual learning
        haze_residual = torch.sigmoid(
            self.final(d1)
        )

        clean = torch.clamp(
            x - haze_residual,
            0,
            1
        )

        return clean


# ============================================================
# 10. CREATE MODEL
# ============================================================

print("\nCreating model...")

model = UNetDenseAttention().to(DEVICE)


# ============================================================
# 11. LOAD BEST MODEL
# ============================================================

print("\nLoading best_model.pth...")

state_dict = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

model.load_state_dict(state_dict)

model.eval()

print("Best model loaded successfully!")


# ============================================================
# 12. CREATE OUTPUT FOLDERS
# ============================================================

INPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "input_hazy"
)

RESTORED_DIR = os.path.join(
    OUTPUT_DIR,
    "restored"
)

GT_DIR = os.path.join(
    OUTPUT_DIR,
    "ground_truth"
)

COMPARISON_DIR = os.path.join(
    OUTPUT_DIR,
    "comparisons"
)

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(RESTORED_DIR, exist_ok=True)
os.makedirs(GT_DIR, exist_ok=True)
os.makedirs(COMPARISON_DIR, exist_ok=True)


# ============================================================
# 13. TENSOR TO PIL IMAGE
# ============================================================

def tensor_to_pil(tensor):

    tensor = (
        tensor
        .detach()
        .cpu()
        .clamp(0, 1)
    )

    return transforms.ToPILImage()(tensor)


# ============================================================
# 14. GENERATE OUTPUTS
# ============================================================

total_psnr = 0.0
total_ssim = 0.0
count = 0

print("\n==============================================")
print("STARTING IMAGE RESTORATION")
print("==============================================")

with torch.no_grad():

    for idx, (input_image, ground_truth, filename) in enumerate(test_loader):

        # Move input to device
        input_device = input_image.to(DEVICE)

        # Run trained model
        restored = model(input_device)

        restored = restored.clamp(0, 1)

        # ----------------------------------------------------
        # Calculate PSNR
        # ----------------------------------------------------

        pred_np = (
            restored[0]
            .cpu()
            .numpy()
            .transpose(1, 2, 0)
        )

        gt_np = (
            ground_truth[0]
            .numpy()
            .transpose(1, 2, 0)
        )

        psnr_value = sk_psnr(
            gt_np,
            pred_np,
            data_range=1.0
        )

        # ----------------------------------------------------
        # Calculate SSIM
        # ----------------------------------------------------

        ssim_value = sk_ssim(
            gt_np,
            pred_np,
            data_range=1.0,
            channel_axis=2
        )

        total_psnr += psnr_value
        total_ssim += ssim_value
        count += 1

        # ----------------------------------------------------
        # Convert to PIL
        # ----------------------------------------------------

        input_pil = tensor_to_pil(
            input_image[0]
        )

        restored_pil = tensor_to_pil(
            restored[0]
        )

        gt_pil = tensor_to_pil(
            ground_truth[0]
        )

        # ----------------------------------------------------
        # Create file name
        # ----------------------------------------------------

        base_name = os.path.splitext(
            filename[0]
        )[0]

        # ----------------------------------------------------
        # Save input
        # ----------------------------------------------------

        input_pil.save(
            os.path.join(
                INPUT_DIR,
                base_name + "_input.png"
            )
        )

        # ----------------------------------------------------
        # Save restored
        # ----------------------------------------------------

        restored_pil.save(
            os.path.join(
                RESTORED_DIR,
                base_name + "_restored.png"
            )
        )

        # ----------------------------------------------------
        # Save ground truth
        # ----------------------------------------------------

        gt_pil.save(
            os.path.join(
                GT_DIR,
                base_name + "_ground_truth.png"
            )
        )

        # ----------------------------------------------------
        # Create side-by-side comparison
        # ----------------------------------------------------

        width, height = input_pil.size

        comparison = Image.new(
            "RGB",
            (
                width * 3,
                height
            )
        )

        comparison.paste(
            input_pil,
            (0, 0)
        )

        comparison.paste(
            restored_pil,
            (width, 0)
        )

        comparison.paste(
            gt_pil,
            (width * 2, 0)
        )

        comparison.save(
            os.path.join(
                COMPARISON_DIR,
                base_name + "_comparison.png"
            )
        )

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (idx + 1) % 100 == 0:

            print(
                f"Processed {idx + 1} / "
                f"{len(test_dataset)} images"
            )


# ============================================================
# 15. FINAL METRICS
# ============================================================

average_psnr = total_psnr / count
average_ssim = total_ssim / count


print("\n==============================================")
print("FINAL RESULTS")
print("==============================================")

print(
    f"Average PSNR : {average_psnr:.4f} dB"
)

print(
    f"Average SSIM : {average_ssim:.4f}"
)

print(
    f"Total images : {count}"
)


# ============================================================
# 16. OUTPUT LOCATIONS
# ============================================================

print("\n==============================================")
print("OUTPUT LOCATIONS")
print("==============================================")

print("\nInput images:")
print(INPUT_DIR)

print("\nRestored images:")
print(RESTORED_DIR)

print("\nGround truth images:")
print(GT_DIR)

print("\nComparison images:")
print(COMPARISON_DIR)

print("\n==============================================")
print("COMPLETED SUCCESSFULLY")
print("==============================================")

Device: cuda

Best model:
/kaggle/input/datasets/lakshmiksr123/result6/best_model.pth

Test hazy folder:
/kaggle/input/datasets/lakshmiksr123/reside/RESIDE-6K/test/hazy

Test GT folder:
/kaggle/input/datasets/lakshmiksr123/reside/RESIDE-6K/test/GT

All paths found successfully!

Input images : 1500
GT images    : 1500

Test dataset ready.
Total test images: 1500

Creating model...

Loading best_model.pth...
Best model loaded successfully!

STARTING IMAGE RESTORATION
Processed 100 / 1500 images
Processed 200 / 1500 images
Processed 300 / 1500 images
Processed 400 / 1500 images
Processed 500 / 1500 images
Processed 600 / 1500 images
Processed 700 / 1500 images
Processed 800 / 1500 images
Processed 900 / 1500 images
Processed 1000 / 1500 images
Processed 1100 / 1500 images
Processed 1200 / 1500 images
Processed 1300 / 1500 images
Processed 1400 / 1500 images
Processed 1500 / 1500 images

FINAL RESULTS
Average PSNR : 28.2767 dB
Average SSIM : 0.9536
Total images : 1500

OUTPUT LOCATIONS

I